# 05 PSO Hyperparameter Tuning (All Datasets)

This notebook runs the clean evaluation pipeline for **all three datasets** (China, COCOMO-81, Desharnais).

**Key features:**
- Leakage-free `X_main` / `X_holdout` split before any scaler fit, PSO tuning, or CV
- Fold-local `StandardScaler` fits inside every CV split
- Baseline and PSO-tuned CNN + MLP models
- EarlyStopping + ReduceLROnPlateau callbacks with internal validation split
- CNN+PSO and MLP+PSO ensemble holdout evaluation
- Standardized metric tables and saved models under `results/metrics/` and `models/`

In [1]:
from importlib import import_module
import importlib.util
import random
from pathlib import Path
import subprocess
import sys
import warnings

import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")

drive = None
if importlib.util.find_spec("google.colab") is not None:
    drive = import_module("google.colab.drive")
    IN_COLAB = True
else:
    IN_COLAB = False


def resolve_project_root() -> Path:
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive")
        candidate_roots = [
            drive_root / "Software-cost-Estimation",
            drive_root / "Colab Notebooks" / "Software-cost-Estimation",
        ]
        for candidate in candidate_roots:
            if (candidate / "src").exists():
                return candidate

        for src_dir in drive_root.rglob("src"):
            if src_dir.is_dir() and src_dir.parent.name == "Software-cost-Estimation":
                return src_dir.parent

        raise FileNotFoundError(
            "Could not find the 'Software-cost-Estimation' project folder in Google Drive. "
            "Upload the full project folder to MyDrive or Colab Notebooks."
        )

    root_dir = Path.cwd()
    if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
        root_dir = root_dir.parent
    return root_dir


root_dir = resolve_project_root()
if not (root_dir / "src").exists():
    raise FileNotFoundError(f"Could not find project root containing 'src' directory. Checked: {root_dir}")

if IN_COLAB and importlib.util.find_spec("pyswarms") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyswarms>=1.3"])

sys.path.insert(0, str(root_dir))
print("Using project root:", root_dir)

from src.cv_pipeline import DATASET_DISPLAY_NAMES, save_benchmark_artifacts
from src.data_loader import load_all_raw_datasets

metrics_dir = root_dir / "results" / "metrics"
models_dir = root_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

raw_datasets = load_all_raw_datasets()
for dataset_key in ("china", "cocomo81", "desharnais"):
    print(f"Loaded {DATASET_DISPLAY_NAMES[dataset_key]}: {raw_datasets[dataset_key].shape}")

print(f"\n{len(raw_datasets)} raw datasets ready for clean holdout + CV benchmarking")

Using project root: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation
Loaded China: (499, 19)
Loaded COCOMO81: (63, 19)
Loaded Desharnais: (81, 13)

3 raw datasets ready for clean holdout + CV benchmarking


In [2]:
TRAINING_EPOCHS = 100
TUNING_EPOCHS = 30
N_PARTICLES = 15
PSO_ITERS = 25

print("Benchmark configuration")
print("- training_epochs:", TRAINING_EPOCHS)
print("- tuning_epochs:", TUNING_EPOCHS)
print("- n_particles:", N_PARTICLES)
print("- pso_iters:", PSO_ITERS)

Benchmark configuration
- training_epochs: 100
- tuning_epochs: 30
- n_particles: 15
- pso_iters: 25


In [ ]:
from src.cv_pipeline import run_full_benchmark

print("Running clean holdout + CV benchmark across all datasets...\n")
benchmark_results = run_full_benchmark(
    raw_datasets=raw_datasets,
    use_log_transform=True,
    training_epochs=TRAINING_EPOCHS,
    tuning_epochs=TUNING_EPOCHS,
    n_particles=N_PARTICLES,
    iters=PSO_ITERS,
    verbose=0,
)

holdout_results = benchmark_results["holdout_results"]
full_comparison_final = benchmark_results["full_comparison_final"]

print("All datasets complete.")

Running clean holdout + CV benchmark across all datasets...




2026-04-28 18:32:01,510 - pyswarms.single.global_best - INFO - Optimize for 25 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|25/25, best_cost=6.32e+3
2026-04-28 20:13:08,576 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 6319.775272267604, best pos: [118.03025098   3.27137681  -3.03194417   0.34758667   2.83746493
  34.38253488]
2026-04-28 20:32:30,886 - pyswarms.single.global_best - INFO - Optimize for 25 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|25/25, best_cost=7.28e+3
2026-04-28 21:59:41,241 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 7279.33904613307, best pos: [ -3.54882061   0.23974447   1.19448736 106.37395344  36.8263399 ]
2026-04-28 22:15:19,448 - pyswarms.single.global_best - INFO - Optimize for 25 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|25/25, best_cost=512
2026-04-28 23:29:48,828 

In [ ]:
print("Final holdout results")
print("=" * 70)
display(holdout_results)

print("\n5-fold cross-validation final comparison on X_main")
print("=" * 70)
full_comparison_final

Final holdout results


NameError: name 'holdout_results' is not defined

In [ ]:
saved_paths = save_benchmark_artifacts(
    benchmark_results=benchmark_results,
    metrics_dir=metrics_dir,
    models_dir=models_dir,
)

for label, path in saved_paths.items():
    print(f"Saved {label}: {path}")

NameError: name 'benchmark_results' is not defined